# LLM 실습 과제 - 내풀이 v2

In [ ]:
# =========================
# 문제 1 (난이도: 하)
# =========================
# 텍스트를 토큰화하고 → ID로 변환 → 다시 토큰으로 복원

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
text = "LLM is powerful"

tokens = tokenizer.tokenize(text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)
decoded_tokens = tokenizer.convert_ids_to_tokens(token_ids)

print("[문제1]")
print("tokens:", tokens)
print("token_ids:", token_ids)
print("decoded_tokens:", decoded_tokens)

In [ ]:
# =========================
# 문제 2 (난이도: 하)
# =========================
# Scaled Dot-Product Attention 구현

import torch
import torch.nn.functional as F

Q = torch.randn(2, 4)
K = torch.randn(2, 4)
V = torch.randn(2, 4)

d = Q.size(-1)

scores = torch.matmul(Q, K.t()) / (d ** 0.5)
weights = F.softmax(scores, dim=-1)
output = torch.matmul(weights, V)

print("[문제2]")
print("scores:\n", scores)
print("weights:\n", weights)
print("output:\n", output)

In [ ]:
# =========================
# 문제 3 (난이도: 상)
# =========================
# KV Cache 성능 효과 실험

import time

def attention_compute(x):
    return torch.matmul(x, x.T)

def run_no_cache(seq_len=50):
    x = torch.randn(seq_len, 64)
    start = time.time()
    for i in range(1, seq_len + 1):
        _ = attention_compute(x[:i])
    return time.time() - start

def run_with_cache(seq_len=50):
    cache = []
    start = time.time()
    for i in range(1, seq_len + 1):
        new_token = torch.randn(1, 64)
        cache.append(new_token)
        cached = torch.cat(cache, dim=0)
        # cache 사용: 새 토큰에 대해서만 attention 계산
        _ = attention_compute(cached[-1:].repeat(cached.shape[0], 1))
    return time.time() - start, len(cache)

t1 = run_no_cache()
t2, cache_size = run_with_cache()

print("[문제3]")
print(f"No Cache: {t1:.4f}s")
print(f"With Cache: {t2:.4f}s")
print(f"Cache size: {cache_size}")

In [ ]:
# =========================
# 문제 4 (난이도: 중)
# =========================
# LLM 메시지 구조 작성

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is machine learning?"},
    {"role": "assistant", "content": "Machine learning is a subset of AI that learns patterns from data."}
]

print("[문제4]")
for msg in messages:
    print(f"  {msg['role']}: {msg['content']}")

In [ ]:
# =========================
# 문제 5 (난이도: 중)
# =========================
# Tool schema 정의 + 함수 실행

import json

def get_weather(city):
    return f"{city} sunny"

# tool schema 정의
tool_schema = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get the weather for a city",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "The city name"}
            },
            "required": ["city"]
        }
    }
}

# JSON arguments 파싱 후 함수 실행
raw_args = '{"city": "Seoul"}'
args = json.loads(raw_args)
result = get_weather(**args)

print("[문제5]")
print("tool_schema:", tool_schema)
print("result:", result)

In [ ]:
# =========================
# 문제 6 (난이도: 중)
# =========================
# tool router 구현

def add(a, b):
    return a + b

def multiply(a, b):
    return a * b

def tool_router(name, args):
    funcs = {
        "add": add,
        "multiply": multiply
    }
    if name in funcs:
        return funcs[name](**args)
    else:
        return f"Unknown tool: {name}"

print("[문제6]")
print("add(3,4):", tool_router("add", {"a": 3, "b": 4}))
print("multiply(3,4):", tool_router("multiply", {"a": 3, "b": 4}))

In [7]:
# =========================
# 문제 7 (난이도: 상) - API 필요
# =========================
# Tool Calling 전체 흐름 구현

from openai import OpenAI
import json

client = OpenAI(api_key="YOUR_API_KEY")

def calculator(a, b):
    return a + b

messages = [{"role": "user", "content": "10 + 20 계산해줘"}]

tools = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Add two numbers",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "number"},
                    "b": {"type": "number"}
                },
                "required": ["a", "b"]
            }
        }
    }
]

try:
    # 1) LLM 호출 (tool_call 요청)
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        tools=tools
    )

    msg = response.choices[0].message
    messages.append(msg)

    # 2) tool_call 추출 + 함수 실행
    tool_call = msg.tool_calls[0]
    args = json.loads(tool_call.function.arguments)
    result = calculator(**args)

    # 3) tool 결과를 메시지에 추가
    messages.append({
        "role": "tool",
        "content": str(result),
        "tool_call_id": tool_call.id
    })

    # 4) final 응답 생성
    final = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages
    )

    print("[문제7]", final.choices[0].message.content)
except Exception as e:
    print(f"[문제7] skipped (API 필요): {e}")

[문제7] skipped (API 필요): Error code: 401 - {'error': {'message': 'Incorrect API key provided: YOUR_API_KEY. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}


In [8]:
# =========================
# 문제 8 (난이도: 상) - API 필요
# =========================
# Streaming 응답 처리

try:
    cache = []

    stream = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": "Explain AI"}],
        stream=True
    )

    for chunk in stream:
        delta = chunk.choices[0].delta
        if hasattr(delta, "content") and delta.content:
            cache.append(delta.content)
            # 길이 제한: 100자 초과 시 중단
            if len("".join(cache)) > 100:
                break

    print("[문제8]", "".join(cache))
except Exception as e:
    print(f"[문제8] skipped (API 필요): {e}")

[문제8] skipped (API 필요): Error code: 401 - {'error': {'message': 'Incorrect API key provided: YOUR_API_KEY. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}


In [9]:
# =========================
# 문제 9 (난이도: 상)
# =========================
# 외부 API를 활용한 환율 조회

import requests

BASE_URL = "https://open.er-api.com/v6/latest/"

def get_exchange_rate(base, target):
    try:
        response = requests.get(BASE_URL + base)
        data = response.json()
        if target in data["rates"]:
            return data["rates"][target]
        else:
            return f"Error: {target} not found"
    except Exception as e:
        return f"Error: {e}"

print("[문제9]")
print("USD → KRW:", get_exchange_rate("USD", "KRW"))
print("USD → INVALID:", get_exchange_rate("USD", "INVALID"))

[문제9]


USD → KRW: 1472.46758


USD → INVALID: Error: INVALID not found


In [10]:
# =========================
# 문제 10 (난이도: 상)
# =========================
# 자연어 수식 계산 agent

expr = "3 더하기 5 곱하기 2"

# 연산자 변환
expr = expr.replace("더하기", "+").replace("곱하기", "*")
parts = expr.split()

# 토큰 분리 (숫자와 연산자)
tokens = []
for p in parts:
    if p in ["+", "*"]:
        tokens.append(p)
    else:
        tokens.append(int(p))

# 순차 계산 (왼쪽부터)
result = tokens[0]
i = 1
while i < len(tokens):
    op = tokens[i]
    num = tokens[i + 1]
    if op == "+":
        result = result + num
    elif op == "*":
        result = result * num
    i += 2

print("[문제10]")
print(f"수식: 3 더하기 5 곱하기 2")
print(f"변환: {' '.join(str(t) for t in tokens)}")
print(f"결과: {result}")

[문제10]
수식: 3 더하기 5 곱하기 2
변환: 3 + 5 * 2
결과: 16
